# LeetCode 1828 — Queries on Number of Points Inside a Circle bằng Python

Link đề: https://leetcode.com/problems/queries-on-number-of-points-inside-a-circle/ · Độ khó: **Medium** · Chủ đề: Array, Math, Geometry

Skeleton LeetCode:

```python
class Solution:
    def countPoints(self, points: List[List[int]], queries: List[List[int]]) -> List[int]:
        ...
```

Lộ trình: **duyệt mọi (truy vấn, điểm) với `sqrt`** → **so sánh bình phương khoảng cách bằng số nguyên** `O(Q·P)` (lời giải nộp) → **sắp xếp theo `x` + `bisect` lọc dải** → **lưới đếm + tổng tiền tố theo cột** `O(V² + Q·r)` (trả lời follow-up "tốt hơn `O(n)` mỗi truy vấn") → **vector hóa bằng NumPy**.

## 1. Mục tiêu học

1. Kiểm tra "điểm nằm trong (hoặc trên) đường tròn" bằng **số nguyên**: `dx² + dy² ≤ r²` — không cần căn bậc hai, không lo sai số thực.
2. Phân biệt **trong** (`<`) và **trong hoặc trên biên** (`≤`), và xử lý **điểm trùng tọa độ** (phải đếm đủ bội số).
3. Lọc ứng viên nhanh bằng **sắp xếp + `bisect`** theo một trục.
4. Tận dụng **tọa độ nhỏ** (`0..500`) để tiền xử lý **lưới đếm** và **tổng tiền tố theo cột**, trả lời mỗi truy vấn trong `O(r)` thay vì `O(P)`.
5. Kiểm chứng các cách bằng nhau trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho mảng `points`, `points[i] = [x_i, y_i]` là tọa độ điểm thứ `i` trên mặt phẳng. **Có thể có nhiều điểm trùng tọa độ.**

Cho mảng `queries`, `queries[j] = [x_j, y_j, r_j]` mô tả đường tròn tâm `(x_j, y_j)`, bán kính `r_j`.

Với mỗi truy vấn, đếm **số điểm nằm trong đường tròn**; điểm **trên biên** cũng được tính là ở trong. Trả về mảng `answer` với `answer[j]` là kết quả truy vấn thứ `j`.

**Ví dụ 1:** `points = [[1,3],[3,3],[5,3],[2,2]]`, `queries = [[2,3,1],[4,3,1],[1,1,2]]` → `[3,2,2]`.

**Ví dụ 2:** `points = [[1,1],[2,2],[3,3],[4,4],[5,5]]`, `queries = [[1,2,2],[2,2,2],[4,3,2],[4,3,3]]` → `[2,3,2,4]`.

**Ràng buộc:** `1 ≤ points.length ≤ 500`, `0 ≤ x_i, y_i ≤ 500`, `1 ≤ queries.length ≤ 500`, `0 ≤ x_j, y_j ≤ 500`, `1 ≤ r_j ≤ 500`, mọi tọa độ là số nguyên.

**Follow-up:** trả lời mỗi truy vấn tốt hơn `O(n)` được không?

In [1]:
import math
import random
import time
from bisect import bisect_left, bisect_right
from collections import Counter
from math import isqrt
from typing import List

import numpy as np

examples = [
    ([[1, 3], [3, 3], [5, 3], [2, 2]], [[2, 3, 1], [4, 3, 1], [1, 1, 2]], [3, 2, 2]),
    ([[1, 1], [2, 2], [3, 3], [4, 4], [5, 5]], [[1, 2, 2], [2, 2, 2], [4, 3, 2], [4, 3, 3]], [2, 3, 2, 4]),
]
for pts, qs, want in examples:
    print(pts, qs, "->", want)

[[1, 3], [3, 3], [5, 3], [2, 2]] [[2, 3, 1], [4, 3, 1], [1, 1, 2]] -> [3, 2, 2]
[[1, 1], [2, 2], [3, 3], [4, 4], [5, 5]] [[1, 2, 2], [2, 2, 2], [4, 3, 2], [4, 3, 3]] -> [2, 3, 2, 4]


## 3. Kiến thức nền

### 3.1 Khoảng cách Euclid và bình phương khoảng cách

Khoảng cách từ `(px, py)` tới tâm `(cx, cy)` là `√(dx² + dy²)` với `dx = px − cx`, `dy = py − cy`. Vì cả hai vế không âm, `√(dx² + dy²) ≤ r` ⇔ `dx² + dy² ≤ r²` — chỉ dùng **số nguyên**.

In [2]:
px, py, cx, cy, r = 3, 3, 2, 3, 1
dx, dy = px - cx, py - cy
print("dx² + dy² =", dx * dx + dy * dy, "| r² =", r * r, "| trong hoặc trên biên:", dx * dx + dy * dy <= r * r)
print("cách dùng sqrt:", math.hypot(dx, dy) <= r)

dx² + dy² = 1 | r² = 1 | trong hoặc trên biên: True
cách dùng sqrt: True


### 3.2 Số thực và sai số

Số thực dấu phẩy động có sai số làm tròn (`0.1 + 0.2 != 0.3`). Với tọa độ nhỏ ở bài này, `math.hypot` của số nguyên thường vẫn cho kết quả đúng, nhưng thói quen tốt là **so sánh bình phương bằng số nguyên** để chắc chắn tuyệt đối.

In [3]:
print(0.1 + 0.2 == 0.3, 0.1 + 0.2)
big = 10**8 + 1
print("sqrt của số lớn có thể lệch:", math.sqrt(big * big + 1) <= big, "| so sánh nguyên:", big * big + 1 <= big * big)

False 0.30000000000000004
sqrt của số lớn có thể lệch: True | so sánh nguyên: False


### 3.3 `math.isqrt` — căn bậc hai nguyên

`isqrt(t)` = số nguyên `s` lớn nhất với `s² ≤ t`, tính **chính xác** cho mọi số nguyên. Dùng để tìm nửa chiều cao của đường tròn tại một cột: các `y` thỏa `dy² ≤ r² − dx²` là `|dy| ≤ isqrt(r² − dx²)`.

In [4]:
r, dx = 5, 3
h = isqrt(r * r - dx * dx)
print(f"tại cột lệch dx={dx}: |dy| ≤ {h} (vì {h}² = {h*h} ≤ {r*r - dx*dx} < {(h+1)**2})")

tại cột lệch dx=3: |dy| ≤ 4 (vì 4² = 16 ≤ 16 < 25)


### 3.4 Sắp xếp + `bisect` để lấy dải `x ∈ [lo, hi]`

Trên list `xs` đã sắp xếp: các chỉ số `bisect_left(xs, lo) .. bisect_right(xs, hi) − 1` là đúng các phần tử trong đoạn `[lo, hi]`.

In [5]:
xs = sorted([1, 3, 5, 2, 3, 8])
lo, hi = 2, 5
a, b = bisect_left(xs, lo), bisect_right(xs, hi)
print(xs, "-> phần tử trong [2,5]:", xs[a:b])

[1, 2, 3, 3, 5, 8] -> phần tử trong [2,5]: [2, 3, 3, 5]


### 3.5 Tổng tiền tố theo cột

Lưới `cnt[x][y]` = số điểm tại `(x, y)`. Tiền tố cột `pre[x][y] = cnt[x][0] + … + cnt[x][y−1]` cho số điểm trên cột `x` với `y ∈ [a, b]` là `pre[x][b+1] − pre[x][a]` trong `O(1)`.

In [6]:
col = [0, 2, 0, 1, 3]                # cnt[x][y] với y = 0..4
pre = [0]
for v in col:
    pre.append(pre[-1] + v)
print("pre =", pre, "| số điểm y ∈ [1, 3]:", pre[4] - pre[1])

pre = [0, 0, 2, 2, 3, 6] | số điểm y ∈ [1, 3]: 3


### 3.6 NumPy broadcast

Với mảng `X` shape `(P,)` và `CX` shape `(Q, 1)`, `X − CX` có shape `(Q, P)` — tính mọi cặp (truy vấn, điểm) một lúc bằng mã C.

In [7]:
X = np.array([1, 3, 5]); CX = np.array([[2], [4]])
print(X - CX)

[[-1  1  3]
 [-3 -1  1]]


## 4. Ý tưởng nền tảng / trực giác

Tính tay Ví dụ 1, truy vấn `[2,3,1]` (tâm `(2,3)`, `r = 1`, `r² = 1`):

| điểm | dx | dy | dx² + dy² | ≤ 1? |
|---|---|---|---|---|
| (1,3) | −1 | 0 | 1 | ✔ (trên biên) |
| (3,3) | 1 | 0 | 1 | ✔ (trên biên) |
| (5,3) | 3 | 0 | 9 | ✘ |
| (2,2) | 0 | −1 | 1 | ✔ (trên biên) |

Kết quả 3. ✔ Lưu ý cả ba điểm đều **trên biên** — nếu dùng `<` sẽ ra 0!

Truy vấn `[1,1,2]` (`r² = 4`): `(1,3)` có `0 + 4 = 4` ✔, `(3,3)` có `4 + 4 = 8` ✘, `(5,3)` ✘, `(2,2)` có `1 + 1 = 2` ✔ ⇒ 2. ✔

In [8]:
pts = examples[0][0]
for cx, cy, r in examples[0][1]:
    inside = [(px, py) for px, py in pts if (px - cx) ** 2 + (py - cy) ** 2 <= r * r]
    print(f"tâm ({cx},{cy}) r={r}: {inside} -> {len(inside)}")

tâm (2,3) r=1: [(1, 3), (3, 3), (2, 2)] -> 3
tâm (4,3) r=1: [(3, 3), (5, 3)] -> 2
tâm (1,1) r=2: [(1, 3), (2, 2)] -> 2


## 5. Cách 1 — Brute Force / baseline: duyệt mọi cặp, dùng khoảng cách thực

In [9]:
def count_points_float(points, queries):
    ans = []
    for cx, cy, r in queries:
        cnt = 0
        for px, py in points:
            if math.hypot(px - cx, py - cy) <= r:     # khoảng cách thực
                cnt += 1
        ans.append(cnt)
    return ans

for pts, qs, want in examples:
    print(count_points_float(pts, qs), "| kỳ vọng", want)

[3, 2, 2] | kỳ vọng [3, 2, 2]
[2, 3, 2, 4] | kỳ vọng [2, 3, 2, 4]


**Độ phức tạp:** `O(Q · P)` = `500 · 500 = 2.5·10⁵` phép — thực ra **đủ nhanh** cho ràng buộc. Hai điểm yếu: (1) phụ thuộc số thực (về nguyên tắc có thể sai ở biên với tọa độ lớn); (2) không trả lời được follow-up — mỗi truy vấn vẫn `O(P)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (số nguyên):** `dist ≤ r` ⇔ `dx² + dy² ≤ r²` (bình phương hai vế không âm giữ nguyên thứ tự). Mọi đại lượng là số nguyên ⇒ so sánh **chính xác**.

**Insight 2 (lọc theo một trục):** điểm trong hình tròn phải có `|dx| ≤ r` ⇒ chỉ cần xét các điểm có `x ∈ [cx − r, cx + r]`. Sắp xếp điểm theo `x` rồi `bisect` lấy dải — giảm số điểm phải kiểm khi `r` nhỏ.

**Insight 3 (tọa độ nhỏ ⇒ lưới):** tọa độ chỉ `0..500` ⇒ đếm số điểm tại từng ô, dựng **tiền tố theo cột**. Với mỗi cột `x ∈ [cx − r, cx + r] ∩ [0, 500]`: các `y` hợp lệ là `[cy − h, cy + h] ∩ [0, 500]` với `h = isqrt(r² − (x − cx)²)` ⇒ cộng `pre[x][hi+1] − pre[x][lo]`. Mỗi truy vấn `O(r)` ≤ `O(1001)`, không phụ thuộc số điểm `P` — trả lời follow-up.

**Quy tắc biên:** *nếu* `dx² + dy² == r²` *thì* điểm **được tính** (dùng `≤`). *Nếu* có điểm trùng *thì* đếm đủ (lưới lưu **số lượng**, không lưu cờ có/không).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — So sánh bình phương bằng số nguyên (`O(Q·P)`) — lời giải nộp

In [10]:
class SolutionInt:
    def countPoints(self, points: List[List[int]], queries: List[List[int]]) -> List[int]:
        ans = []
        for cx, cy, r in queries:
            r2 = r * r
            ans.append(sum((px - cx) * (px - cx) + (py - cy) * (py - cy) <= r2 for px, py in points))
        return ans

for pts, qs, want in examples:
    print(SolutionInt().countPoints(pts, qs) == want)

True
True


`sum(điều_kiện for …)` cộng các giá trị `True` (= 1) — cách viết gọn để đếm. **Độ phức tạp:** `O(Q·P)` thời gian, `O(1)` bộ nhớ phụ.

### 7.2 Cách 3 — Sắp xếp theo `x` + `bisect` lọc dải

Sắp xếp các điểm theo `x` một lần (`O(P log P)`). Mỗi truy vấn: `bisect` lấy các điểm có `x ∈ [cx − r, cx + r]`, rồi kiểm điều kiện đầy đủ chỉ trên các điểm đó.

In [11]:
class SolutionStrip:
    def countPoints(self, points: List[List[int]], queries: List[List[int]]) -> List[int]:
        pts = sorted(map(tuple, points))
        xs = [p[0] for p in pts]
        ans = []
        for cx, cy, r in queries:
            a = bisect_left(xs, cx - r)
            b = bisect_right(xs, cx + r)
            r2 = r * r
            ans.append(sum((px - cx) ** 2 + (py - cy) ** 2 <= r2 for px, py in pts[a:b]))
        return ans

for pts, qs, want in examples:
    print(SolutionStrip().countPoints(pts, qs) == want)

True
True


**Độ phức tạp:** worst-case vẫn `O(Q·P)` (khi `r` lớn, dải chứa mọi điểm), nhưng trung bình nhanh hơn khi `r` nhỏ so với miền. Bộ nhớ `O(P)`.

### 7.3 Cách 4 — Lưới đếm + tiền tố theo cột (`O(V² + Q·r)`, `V = 501`)

In [12]:
class SolutionGrid:
    V = 501
    def countPoints(self, points: List[List[int]], queries: List[List[int]]) -> List[int]:
        V = self.V
        cnt = [[0] * V for _ in range(V)]
        for x, y in points:
            cnt[x][y] += 1                         # đếm cả điểm trùng
        pre = []
        for x in range(V):
            row, s = [0], 0
            for y in range(V):
                s += cnt[x][y]; row.append(s)
            pre.append(row)                         # pre[x][y+1] - pre[x][a] = số điểm y ∈ [a, y]
        ans = []
        for cx, cy, r in queries:
            total = 0
            for x in range(max(0, cx - r), min(V - 1, cx + r) + 1):
                h = isqrt(r * r - (x - cx) * (x - cx))
                lo, hi = max(0, cy - h), min(V - 1, cy + h)
                if lo <= hi:
                    total += pre[x][hi + 1] - pre[x][lo]
            ans.append(total)
        return ans

for pts, qs, want in examples:
    print(SolutionGrid().countPoints(pts, qs) == want)

True
True


**Độ phức tạp:** tiền xử lý `O(V²) ≈ 2.5·10⁵`; mỗi truy vấn duyệt tối đa `2r + 1 ≤ 1001` cột, mỗi cột `O(1)` ⇒ `O(Q · r)`. **Không phụ thuộc** số điểm — tốt khi có rất nhiều điểm (ví dụ `10⁶` điểm) và nhiều truy vấn. Bộ nhớ `O(V²)`.

Có thể đổi vai trò: duyệt theo **hàng** thay vì cột, như nhau. Nếu tọa độ lớn (không làm lưới được) thì nén tọa độ + cây phạm vi (range tree / k-d tree) — ngoài phạm vi bài.

### 7.4 Cách 5 — NumPy vector hóa (`O(Q·P)` nhưng chạy bằng C)

In [13]:
class SolutionNumpy:
    def countPoints(self, points: List[List[int]], queries: List[List[int]]) -> List[int]:
        P = np.asarray(points, dtype=np.int64)
        Qa = np.asarray(queries, dtype=np.int64)
        dx = P[None, :, 0] - Qa[:, None, 0]         # shape (Q, P)
        dy = P[None, :, 1] - Qa[:, None, 1]
        inside = dx * dx + dy * dy <= (Qa[:, 2] ** 2)[:, None]
        return inside.sum(axis=1).tolist()

for pts, qs, want in examples:
    print(SolutionNumpy().countPoints(pts, qs) == want)

True
True


Cùng độ phức tạp `O(Q·P)` và bộ nhớ `O(Q·P)` cho ma trận tạm, nhưng nhanh hơn nhiều lần trong Python thuần. (LeetCode Python có NumPy, nhưng không nên phụ thuộc — dùng để thực hành/thực tế.)

## 8. TRACE từng bước

Trace Cách 4 (lưới) cho truy vấn `[4,3,2]` của Ví dụ 2 (điểm trên đường chéo `(1,1)…(5,5)`), `r² = 4`:

| cột x | `dx` | `h = isqrt(4 − dx²)` | khoảng y | điểm trên cột trong khoảng |
|---|---|---|---|---|
| 2 | −2 | 0 | [3, 3] | (2,2)? y=2 ∉ ⇒ 0 |
| 3 | −1 | 1 | [2, 4] | (3,3) ⇒ 1 |
| 4 | 0 | 2 | [1, 5] | (4,4) ⇒ 1 |
| 5 | 1 | 1 | [2, 4] | (5,5)? y=5 ∉ ⇒ 0 |
| 6 | 2 | 0 | [3, 3] | không có ⇒ 0 |

Tổng 2. ✔

In [14]:
def trace_grid(points, query, V=501):
    cx, cy, r = query
    cnt = Counter(map(tuple, points))
    total = 0
    for x in range(max(0, cx - r), min(V - 1, cx + r) + 1):
        h = isqrt(r * r - (x - cx) ** 2)
        lo, hi = max(0, cy - h), min(V - 1, cy + h)
        here = sum(c for (px, py), c in cnt.items() if px == x and lo <= py <= hi)
        total += here
        print(f"cột x={x}: dx={x - cx:>2}, h={h}, y ∈ [{lo}, {hi}] -> {here} điểm (cộng dồn {total})")
    return total

trace_grid(examples[1][0], [4, 3, 2])

cột x=2: dx=-2, h=0, y ∈ [3, 3] -> 0 điểm (cộng dồn 0)
cột x=3: dx=-1, h=1, y ∈ [2, 4] -> 1 điểm (cộng dồn 1)
cột x=4: dx= 0, h=2, y ∈ [1, 5] -> 1 điểm (cộng dồn 2)
cột x=5: dx= 1, h=1, y ∈ [2, 4] -> 0 điểm (cộng dồn 2)
cột x=6: dx= 2, h=0, y ∈ [3, 3] -> 0 điểm (cộng dồn 2)


2

Trace Cách 2 (so sánh nguyên) cho truy vấn `[4,3,3]` (đáp án 4):

In [15]:
def trace_int(points, query):
    cx, cy, r = query
    r2, cnt = r * r, 0
    for px, py in points:
        d2 = (px - cx) ** 2 + (py - cy) ** 2
        ok = d2 <= r2
        cnt += ok
        print(f"điểm ({px},{py}): d² = {d2:>2} {'≤' if ok else '>'} {r2} -> {'tính' if ok else 'bỏ'} (đếm = {cnt})")
    return cnt

trace_int(examples[1][0], [4, 3, 3])

điểm (1,1): d² = 13 > 9 -> bỏ (đếm = 0)
điểm (2,2): d² =  5 ≤ 9 -> tính (đếm = 1)
điểm (3,3): d² =  1 ≤ 9 -> tính (đếm = 2)
điểm (4,4): d² =  1 ≤ 9 -> tính (đếm = 3)
điểm (5,5): d² =  5 ≤ 9 -> tính (đếm = 4)


4

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def countPoints(self, points: List[List[int]], queries: List[List[int]]) -> List[int]:
        ans = []
        for cx, cy, r in queries:
            r2 = r * r
            cnt = 0
            for px, py in points:
                dx, dy = px - cx, py - cy
                if dx * dx + dy * dy <= r2:      # trong hoặc trên biên, số nguyên chính xác
                    cnt += 1
            ans.append(cnt)
        return ans

for pts, qs, want in examples:
    assert Solution().countPoints(pts, qs) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Cách 2:** với `d = √(dx² + dy²) ≥ 0` và `r ≥ 1 > 0`, hàm `t ↦ t²` đồng biến trên `[0, ∞)` nên `d ≤ r ⇔ d² ≤ r²`. Mỗi điểm (kể cả các bản trùng — chúng là các phần tử riêng của list) được xét đúng một lần ⇒ đếm đúng.

**Cách 3:** điểm thỏa `dx² + dy² ≤ r²` thì `dx² ≤ r²` ⇒ `|dx| ≤ r` ⇒ `x ∈ [cx − r, cx + r]`. Vậy dải `bisect` chứa **mọi** điểm cần đếm; ta kiểm lại điều kiện đầy đủ trong dải nên không đếm thừa.

**Cách 4:** tập điểm nguyên trong hình tròn chia theo cột `x`. Tại cột `x` (với `|x − cx| ≤ r`), điều kiện là `(y − cy)² ≤ r² − (x − cx)²` ⇔ `|y − cy| ≤ isqrt(r² − (x − cx)²) = h` (vì `y − cy` nguyên và `isqrt` là căn nguyên chính xác). Cắt với miền `[0, 500]` (ngoài miền không có điểm). Tiền tố cột đếm đúng số điểm (có bội) trong đoạn `y`. Cộng mọi cột ⇒ đúng.

**Cách 5:** tính đúng biểu thức của Cách 2 cho mọi cặp bằng số nguyên 64-bit (giá trị lớn nhất `2 · 500² = 5·10⁵`, không tràn).

## 11. Phân tích Time / Space

Gọi `P = len(points) ≤ 500`, `Q = len(queries) ≤ 500`, `V = 501` (kích thước miền tọa độ), `R = max r ≤ 500`.

| Cách | Time | Space |
|---|---|---|
| 1. Số thực | `O(Q·P)` | `O(1)` |
| 2. Số nguyên | `O(Q·P)` | `O(1)` |
| 3. Sắp xếp + dải | `O(P log P + Q·(log P + k))`, `k` = số điểm trong dải (worst `P`) | `O(P)` |
| 4. Lưới + tiền tố cột | `O(V² + Q·R)` | `O(V²)` |
| 5. NumPy | `O(Q·P)` (bằng C) | `O(Q·P)` |

In [17]:
def gen(P, Q, rng, rmax=500):
    pts = [[rng.randint(0, 500), rng.randint(0, 500)] for _ in range(P)]
    qs = [[rng.randint(0, 500), rng.randint(0, 500), rng.randint(1, rmax)] for _ in range(Q)]
    return pts, qs

rng = random.Random(1828)
cases = [("ràng buộc đề P=Q=500", *gen(500, 500, rng)),
         ("r nhỏ (≤ 20)", *gen(500, 500, rng, 20)),
         ("nhiều điểm P=20000, Q=500 (ngoài đề)", *gen(20000, 500, rng, 100))]
for label, pts, qs in cases:
    res = []
    for name, sol in [("Số nguyên/Nộp", Solution()), ("Dải bisect", SolutionStrip()),
                      ("Lưới", SolutionGrid()), ("NumPy", SolutionNumpy())]:
        t = time.perf_counter(); r = sol.countPoints(pts, qs); res.append(r)
        print(f"[{label}] {name}: {1000*(time.perf_counter()-t):.0f} ms")
    assert all(r == res[0] for r in res)

[ràng buộc đề P=Q=500] Số nguyên/Nộp: 25 ms
[ràng buộc đề P=Q=500] Dải bisect: 18 ms
[ràng buộc đề P=Q=500] Lưới: 40 ms
[ràng buộc đề P=Q=500] NumPy: 4 ms
[r nhỏ (≤ 20)] Số nguyên/Nộp: 23 ms
[r nhỏ (≤ 20)] Dải bisect: 1 ms


[r nhỏ (≤ 20)] Lưới: 9 ms


[r nhỏ (≤ 20)] NumPy: 3 ms


[nhiều điểm P=20000, Q=500 (ngoài đề)] Số nguyên/Nộp: 822 ms
[nhiều điểm P=20000, Q=500 (ngoài đề)] Dải bisect: 165 ms
[nhiều điểm P=20000, Q=500 (ngoài đề)] Lưới: 18 ms


[nhiều điểm P=20000, Q=500 (ngoài đề)] NumPy: 128 ms


Nhận xét: với ràng buộc của đề, cách đơn giản đã đủ nhanh; lưới thắng rõ khi **số điểm rất lớn** (trường hợp thứ ba), vì chi phí mỗi truy vấn chỉ phụ thuộc `r`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu bản số thực)

In [18]:
sols = [Solution(), SolutionInt(), SolutionStrip(), SolutionGrid(), SolutionNumpy()]
edge = examples + [
    ([[0, 0]], [[0, 0, 1]], [1]),                         # điểm trùng tâm
    ([[0, 0], [0, 0], [0, 0]], [[0, 0, 1]], [3]),         # điểm trùng tọa độ: đếm đủ
    ([[3, 4]], [[0, 0, 5]], [1]),                         # đúng trên biên (3-4-5)
    ([[3, 4]], [[0, 0, 4]], [0]),
    ([[500, 500], [0, 0]], [[0, 0, 500], [500, 500, 500], [250, 250, 354]], [1, 1, 2]),
    ([[1, 1], [2, 2]], [[100, 100, 1]], [0]),             # hình tròn không chứa điểm nào
]
for pts, qs, want in edge:
    assert count_points_float(pts, qs) == want
    for s in sols:
        assert s.countPoints(pts, qs) == want, (pts, qs, s)
    print("OK", pts[:3], qs, "->", want)

OK [[1, 3], [3, 3], [5, 3]] [[2, 3, 1], [4, 3, 1], [1, 1, 2]] -> [3, 2, 2]
OK [[1, 1], [2, 2], [3, 3]] [[1, 2, 2], [2, 2, 2], [4, 3, 2], [4, 3, 3]] -> [2, 3, 2, 4]
OK [[0, 0]] [[0, 0, 1]] -> [1]
OK [[0, 0], [0, 0], [0, 0]] [[0, 0, 1]] -> [3]
OK [[3, 4]] [[0, 0, 5]] -> [1]
OK [[3, 4]] [[0, 0, 4]] -> [0]
OK [[500, 500], [0, 0]] [[0, 0, 500], [500, 500, 500], [250, 250, 354]] -> [1, 1, 2]
OK [[1, 1], [2, 2]] [[100, 100, 1]] -> [0]


(`250² + 250² = 125000 ≤ 354² = 125316` nên cả `(0,0)` và `(500,500)` nằm trong hình tròn tâm `(250,250)` bán kính 354 — đã đối chiếu bằng bản số thực.)

### 12.2 Ngẫu nhiên có seed, đúng ràng buộc

In [19]:
rng = random.Random(42)
for _ in range(300):
    P, Q = rng.randint(1, 60), rng.randint(1, 30)
    span = rng.choice([5, 30, 500])
    pts = [[rng.randint(0, span), rng.randint(0, span)] for _ in range(P)]
    qs = [[rng.randint(0, span), rng.randint(0, span), rng.randint(1, max(1, span))] for _ in range(Q)]
    want = count_points_float(pts, qs)
    for s in sols:
        assert s.countPoints(pts, qs) == want, (pts, qs)
print("300 bộ dữ liệu ngẫu nhiên: 5 cách khớp bản số thực")

300 bộ dữ liệu ngẫu nhiên: 5 cách khớp bản số thực


### 12.3 Kiểm tra riêng các điểm nằm **đúng trên biên**

Sinh các bộ ba Pythagore và điểm `(cx + a, cy + b)` với `a² + b² = r²` — đều phải được đếm.

In [20]:
boundary = []
for r in range(1, 60):
    for a in range(0, r + 1):
        b = isqrt(r * r - a * a)
        if a * a + b * b == r * r:                 # (a, b) nằm đúng trên đường tròn bán kính r
            boundary.append((a, b, r))
cnt_ok = 0
for a, b, r in boundary[:150]:
    cx, cy = 100, 100
    pts = [[cx + a, cy + b], [cx - a, cy - b], [cx + a, cy - b]]
    for s in sols:
        assert s.countPoints(pts, [[cx, cy, r]]) == [3]
    cnt_ok += 1
print("đã kiểm", cnt_ok, "bộ điểm trên biên, ví dụ:", boundary[:6])

đã kiểm 150 bộ điểm trên biên, ví dụ: [(0, 1, 1), (1, 0, 1), (0, 2, 2), (2, 0, 2), (0, 3, 3), (3, 0, 3)]


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `<` (bỏ điểm trên biên)

In [21]:
def wrong_strict(points, queries):
    return [sum((px - cx) ** 2 + (py - cy) ** 2 < r * r for px, py in points) for cx, cy, r in queries]   # SAI

pts, qs, want = examples[0]
print("sai:", wrong_strict(pts, qs), "| đúng:", want)

sai: [0, 0, 1] | đúng: [3, 2, 2]


**Vì sao sai:** đề nói "điểm trên biên được coi là ở trong". Ví dụ 1 truy vấn đầu, cả 3 điểm đều nằm **đúng trên** đường tròn. **Sửa:** `<=`.

### Lỗi 2 — Dùng khoảng cách Manhattan hoặc "hình vuông bao"

In [22]:
def wrong_manhattan(points, queries):
    return [sum(abs(px - cx) + abs(py - cy) <= r for px, py in points) for cx, cy, r in queries]   # SAI: hình thoi

def wrong_square(points, queries):
    return [sum(max(abs(px - cx), abs(py - cy)) <= r for px, py in points) for cx, cy, r in queries]  # SAI: hình vuông

for pts, qs in [examples[1][:2], ([[3, 4], [2, 2]], [[0, 0, 5], [0, 0, 3]])]:
    print("Manhattan:", wrong_manhattan(pts, qs), "| hình vuông:", wrong_square(pts, qs), "| đúng:", Solution().countPoints(pts, qs))

Manhattan: [2, 3, 2, 4] | hình vuông: [3, 4, 4, 5] | đúng: [2, 3, 2, 4]
Manhattan: [1, 0] | hình vuông: [2, 1] | đúng: [2, 1]


**Vì sao sai:** trên Ví dụ 2 bản Manhattan **tình cờ** ra đúng (các điểm nằm trên đường chéo), nhưng với bộ tự dựng thứ hai (điểm `(3,4)` cách gốc đúng 5) thì lộ lỗi. `|dx| + |dy| ≤ r` là **hình thoi** nằm trong hình tròn (đếm thiếu); `max(|dx|, |dy|) ≤ r` là **hình vuông** bao ngoài (đếm thừa). Hình vuông chỉ dùng để **lọc sơ bộ** (như Cách 3), sau đó phải kiểm `dx² + dy² ≤ r²`.

### Lỗi 3 — Khử trùng điểm bằng `set`

In [23]:
def wrong_dedup(points, queries):
    uniq = {tuple(p) for p in points}                    # SAI: điểm trùng bị gộp
    return [sum((px - cx) ** 2 + (py - cy) ** 2 <= r * r for px, py in uniq) for cx, cy, r in queries]

pts, qs = [[0, 0], [0, 0], [1, 0]], [[0, 0, 1]]
print("sai:", wrong_dedup(pts, qs), "| đúng:", Solution().countPoints(pts, qs))

sai: [2] | đúng: [3]


**Vì sao sai:** đề nói rõ "nhiều điểm có thể trùng tọa độ" — mỗi điểm là một phần tử riêng. Nếu muốn gộp để tăng tốc thì dùng `Counter` và cộng **số lượng**.

### Lỗi 4 — Lưới không cắt biên: chỉ số âm âm thầm "quay vòng"

In [24]:
def wrong_grid_no_clip(points, queries, V=501):
    cnt = [[0] * V for _ in range(V)]
    for x, y in points:
        cnt[x][y] += 1
    pre = []
    for x in range(V):
        row = [0]
        for y in range(V): row.append(row[-1] + cnt[x][y])
        pre.append(row)
    ans = []
    for cx, cy, r in queries:
        total = 0
        for x in range(cx - r, cx + r + 1):                 # SAI: không cắt về [0, 500]
            if x >= V:
                continue
            h = isqrt(r * r - (x - cx) ** 2)
            lo, hi = cy - h, min(V - 1, cy + h)             # SAI: lo có thể âm
            total += pre[x][hi + 1] - pre[x][lo]
        ans.append(total)
    return ans

pts, qs = [[500, 0], [0, 3]], [[0, 0, 3]]
print("sai:", wrong_grid_no_clip(pts, qs), "| đúng:", Solution().countPoints(pts, qs))

sai: [0] | đúng: [1]


**Vì sao sai:** với tâm `(0, 0)` và `r = 3`, cột `x = −1, −2, −3` trong Python **không báo lỗi** mà trỏ tới `pre[500], pre[499], …` (cột cuối lưới, nơi có điểm `(500, 0)`); `lo` âm làm `pre[x][lo]` đọc từ **cuối hàng** (tổng gần như cả cột) nên phép trừ ra số âm. Các sai số cộng trừ lẫn nhau cho kết quả vô nghĩa (ở đây 0 thay vì 1) — và **không có exception nào** cảnh báo. **Sửa:** `max(0, ·)` / `min(V − 1, ·)` cho cả `x` lẫn `y`, như Cách 4.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Điểm trong hình tròn"** ⇒ `dx² + dy² ≤ r²` với số nguyên; nhớ `≤` nếu biên tính là trong.
- **Ràng buộc nhỏ** (`P, Q ≤ 500`) ⇒ `O(P·Q)` là đủ, đừng tối ưu sớm.
- **Follow-up "tốt hơn `O(n)` mỗi truy vấn"** + tọa độ nhỏ ⇒ lưới đếm + tiền tố (theo cột/hàng); tọa độ lớn ⇒ nén tọa độ, k-d tree, hoặc xử lý offline.
- Lọc sơ bộ bằng hình vuông bao / dải `x` rồi kiểm chính xác.
- Python: chỉ số âm **không** ném lỗi — luôn cắt biên khi truy cập lưới.

## 15. Biến thể của bài

1. **Điểm trong hình chữ nhật** (truy vấn `[x1,y1,x2,y2]`) ⇒ tiền tố 2 chiều, mỗi truy vấn `O(1)` (Bài tập 5 có liên quan).
2. **Điểm trong vành khuyên** `r1 ≤ d ≤ r2` ⇒ `đếm(≤ r2) − đếm(< r1)` (Bài tập 5).
3. **Đếm điểm nguyên (lattice points) trong hình tròn** không có danh sách điểm: LeetCode 2249 *Count Lattice Points Inside a Circle*.
4. **Nhiều điểm, nhiều truy vấn, tọa độ lớn** ⇒ k-d tree / phân lô theo lưới thô.
5. **Hình tròn chứa nhiều điểm nhất với bán kính cố định:** LeetCode 1453 *Maximum Number of Darts Inside of a Circular Dartboard*.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Số thực | `hypot(dx, dy) ≤ r` | `O(QP)` | `O(1)` | minh họa, không khuyến khích |
| 2. Số nguyên | `dx² + dy² ≤ r²` | `O(QP)` | `O(1)` | **nộp bài** |
| 3. Dải `x` + bisect | lọc theo `x` rồi kiểm | `O(QP)` worst, nhanh khi `r` nhỏ | `O(P)` | `r` nhỏ so với miền |
| 4. Lưới + tiền tố cột | đếm theo cột với `isqrt` | `O(V² + QR)` | `O(V²)` | **rất nhiều điểm**, tọa độ nhỏ (follow-up) |
| 5. NumPy | broadcast mọi cặp | `O(QP)` bằng C | `O(QP)` | thực tế/khoa học dữ liệu |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `points = [[0,0],[1,1],[2,2]]`, `queries = [[0,0,1],[1,1,1],[0,0,3]]`?

**Bài 2 (tính tay):** Tính `dx² + dy²` cho truy vấn `[2,2,2]` của Ví dụ 2 với từng điểm và suy ra kết quả 3.

**Bài 3 (điền TODO):** Viết `count_on_border(points, queries)` — đếm số điểm nằm **đúng trên** đường tròn.

**Bài 4 (tự viết lại):** Viết `count_points_counter(points, queries)` gộp điểm trùng bằng `Counter` và cộng **trọng số** — nhanh hơn khi có nhiều điểm trùng.

**Bài 5 (mini challenge):** Viết `count_annulus(points, queries)` với truy vấn `[cx, cy, r1, r2]` (`0 ≤ r1 ≤ r2`): đếm điểm có `r1 ≤ d ≤ r2`, dùng lưới + tiền tố cột (tái sử dụng ý tưởng Cách 4), đối chiếu với cách trực tiếp.

In [25]:
def count_on_border(points, queries):
    # TODO
    pass

def count_points_counter(points, queries):
    # TODO
    pass

def count_annulus(points, queries):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | `(1,1)` cách `(0,0)` là `√2 > 1` | Truy vấn 2: `(0,0)` và `(2,2)` cách `√2` | Kết quả `[1, 1, 3]` |
| 2 | Tâm `(2,2)`, `r² = 4` | `(1,1)`: 2, `(2,2)`: 0, `(3,3)`: 2 | `(4,4)`: 8 > 4 ⇒ 3 điểm |
| 3 | Đổi `<=` thành `==` | Vẫn dùng số nguyên | `sum(d2 == r*r ...)` |
| 4 | `Counter(map(tuple, points))` | Duyệt `(điểm, số lượng)` | Cộng `số lượng` khi thỏa |
| 5 | `d ≥ r1` ⇔ không nằm **trong ngặt** hình tròn `r1` | `đếm(d² ≤ r2²) − đếm(d² < r1²) = đếm(d² ≤ r2²) − đếm(d² ≤ r1² − 1)` | Hàm lưới nhận "ngưỡng `t`" thay cho `r²`: `h = isqrt(t − dx²)` |

## 19. Đáp án tham khảo

In [26]:
r1 = Solution().countPoints([[0, 0], [1, 1], [2, 2]], [[0, 0, 1], [1, 1, 1], [0, 0, 3]])
print("Bài 1:", r1); assert r1 == [1, 1, 3]
print("Bài 2:"); assert trace_int(examples[1][0], [2, 2, 2]) == 3

Bài 1: [1, 1, 3]
Bài 2:
điểm (1,1): d² =  2 ≤ 4 -> tính (đếm = 1)
điểm (2,2): d² =  0 ≤ 4 -> tính (đếm = 2)
điểm (3,3): d² =  2 ≤ 4 -> tính (đếm = 3)
điểm (4,4): d² =  8 > 4 -> bỏ (đếm = 3)
điểm (5,5): d² = 18 > 4 -> bỏ (đếm = 3)


In [27]:
def count_on_border(points, queries):
    return [sum((px - cx) ** 2 + (py - cy) ** 2 == r * r for px, py in points) for cx, cy, r in queries]

def count_points_counter(points, queries):
    cnt = Counter(map(tuple, points))
    return [sum(c for (px, py), c in cnt.items() if (px - cx) ** 2 + (py - cy) ** 2 <= r * r)
            for cx, cy, r in queries]

assert count_on_border(*examples[0][:2]) == [3, 2, 1]
rng = random.Random(9)
for _ in range(300):
    pts = [[rng.randint(0, 12), rng.randint(0, 12)] for _ in range(rng.randint(1, 40))]
    qs = [[rng.randint(0, 12), rng.randint(0, 12), rng.randint(1, 8)] for _ in range(10)]
    assert count_points_counter(pts, qs) == Solution().countPoints(pts, qs)
    assert count_on_border(pts, qs) == [sum(math.isclose(math.hypot(px - cx, py - cy), r) for px, py in pts) for cx, cy, r in qs]
print("Bài 3, 4: OK", count_on_border(*examples[0][:2]))

Bài 3, 4: OK [3, 2, 1]


In [28]:
def build_prefix(points, V=501):
    cnt = [[0] * V for _ in range(V)]
    for x, y in points:
        cnt[x][y] += 1
    pre = []
    for x in range(V):
        row = [0]
        for y in range(V): row.append(row[-1] + cnt[x][y])
        pre.append(row)
    return pre

def count_le(pre, cx, cy, t, V=501):
    """số điểm có dx² + dy² <= t (t có thể âm -> 0)."""
    if t < 0:
        return 0
    R = isqrt(t); total = 0
    for x in range(max(0, cx - R), min(V - 1, cx + R) + 1):
        h = isqrt(t - (x - cx) ** 2)
        lo, hi = max(0, cy - h), min(V - 1, cy + h)
        if lo <= hi:
            total += pre[x][hi + 1] - pre[x][lo]
    return total

def count_annulus(points, queries):
    pre = build_prefix(points)
    return [count_le(pre, cx, cy, r2 * r2) - count_le(pre, cx, cy, r1 * r1 - 1) for cx, cy, r1, r2 in queries]

def count_annulus_direct(points, queries):
    return [sum(r1 * r1 <= (px - cx) ** 2 + (py - cy) ** 2 <= r2 * r2 for px, py in points)
            for cx, cy, r1, r2 in queries]

rng = random.Random(30)
pts = [[rng.randint(0, 500), rng.randint(0, 500)] for _ in range(400)] + [[250, 250]] * 3
qs = []
for _ in range(60):
    r1 = rng.randint(0, 200); r2 = rng.randint(r1, 300)
    qs.append([rng.randint(0, 500), rng.randint(0, 500), r1, r2])
qs.append([250, 250, 0, 0])                       # vành khuyên suy biến: chỉ tâm
assert count_annulus(pts, qs) == count_annulus_direct(pts, qs)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK; truy vấn [250,250,0,0] ->", count_annulus(pts, [[250, 250, 0, 0]]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK; truy vấn [250,250,0,0] -> [3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = []
for cx, cy, r in queries:
    r2 = r * r
    ans.append(sum((px - cx) ** 2 + (py - cy) ** 2 <= r2 for px, py in points))
return ans

# follow-up (tọa độ 0..500): tiền tố theo cột
#   với x trong [cx-r, cx+r] ∩ [0,500]: h = isqrt(r*r - (x-cx)**2)
#   cộng pre[x][min(500, cy+h) + 1] - pre[x][max(0, cy-h)]
```

**Checklist:**
- [ ] So sánh `dx² + dy² <= r²` bằng **số nguyên** (không `sqrt`)?
- [ ] Dấu `<=` (điểm trên biên tính là trong)?
- [ ] Không khử trùng điểm (hoặc nếu gộp thì cộng số lượng)?
- [ ] Nếu dùng lưới: cắt biên `x`, `y` về `[0, 500]` (chỉ số âm không báo lỗi)?
- [ ] Trả về list theo đúng thứ tự truy vấn.